# Dataset Preparation

In [ ]:
from datasets import load_dataset, Dataset

raw = load_dataset("DeepMount00/CulturaViva-ITA", split="train")
split_ds = raw.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"]
test_ds = split_ds["test"]


def build_ir_split(ds, split_name):
    queries = []
    corpus = []
    qrels = []

    for i, row in enumerate(ds):
        qid = f"{split_name}_q_{i}"
        did = f"{split_name}_d_{i}"

        queries.append(
            {
                "_id": qid,
                "text": row["prompt"],
            }
        )

        corpus.append(
            {
                "_id": did,
                "title": "",
                "text": row["answer"],
            }
        )

        qrels.append(
            {
                "query-id": qid,
                "corpus-id": did,
                "score": 1,
            }
        )

    return (
        Dataset.from_list(queries),
        Dataset.from_list(corpus),
        Dataset.from_list(qrels),
    )


queries_train, corpus_train, qrels_train = build_ir_split(train_ds, "train")
queries_test, corpus_test, qrels_test = build_ir_split(test_ds, "test")

repo_id = "lopozz/CulturaViva-Retrieval"

# push to repo
# DatasetDict({
#     "train": queries_train,
#     "test": queries_test,
# }).push_to_hub(repo_id, config_name="queries")

# # push corpus as one config
# DatasetDict({
#     "train": corpus_train,
#     "test": corpus_test,
# }).push_to_hub(repo_id, config_name="corpus")

# # push qrels as one config
# DatasetDict({
#     "train": qrels_train,
#     "test": qrels_test,
# }).push_to_hub(repo_id, config_name="qrels")

# Train

In [ ]:
from datasets import load_dataset

repo_id = "lopozz/CulturaViva-Retrieval"


def build_pair_dataset(
    repo_id: str, split: str, max_examples: int | None = None
) -> Dataset:
    queries_ds = load_dataset(repo_id, "queries", split=split)
    corpus_ds = load_dataset(repo_id, "corpus", split=split)
    qrels_ds = load_dataset(repo_id, "qrels", split=split)

    query_text_by_id = {row["_id"]: row["text"] for row in queries_ds}
    doc_text_by_id = {row["_id"]: row["text"] for row in corpus_ds}

    pairs = []
    for row in qrels_ds:
        if int(row["score"]) > 0:
            qid = row["query-id"]
            did = row["corpus-id"]

            if qid in query_text_by_id and did in doc_text_by_id:
                pairs.append(
                    {
                        "query": query_text_by_id[qid],
                        "answer": doc_text_by_id[did],
                    }
                )

                if max_examples is not None and len(pairs) >= max_examples:
                    break

    return Dataset.from_list(pairs)


train_dataset = build_pair_dataset(repo_id, "train", max_examples=100)
eval_dataset = build_pair_dataset(repo_id, "test", max_examples=10)

print(train_dataset)
print(eval_dataset)

In [ ]:
import logging

from sentence_transformers import (
    SparseEncoder,
    SparseEncoderModelCardData,
    SparseEncoderTrainer,
    SparseEncoderTrainingArguments,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.losses import (
    SparseMultipleNegativesRankingLoss,
    SpladeLoss,
)
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO
)

# 1. Load a model to finetune with 2. (Optional) model card data
mlm_transformer = Transformer("jhu-clsp/mmBERT-small", transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)


# 4. Define a loss function
loss = SpladeLoss(
    model=model,
    loss=SparseMultipleNegativesRankingLoss(model=model),
    query_regularizer_weight=0,
    document_regularizer_weight=3e-3,
)

batch_size = 1
# 5. (Optional) Specify training arguments
run_name = "inference-free-splade-mmBERT-small-ita"
args = SparseEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    learning_rate_mapping={
        r"0\.sub_modules\.query\.0\.weight": 1e-3
    },  # Set a higher learning rate for the SparseStaticEmbedding module (see https://huggingface.co/blog/train-sparse-encoder#inference-free-splade)
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    router_mapping={
        "query": "query",
        "answer": "document",
    },  # Map the column names to the routes
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
)

# 6. Create a trainer & train
trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

# 9. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

# 10. (Optional) Push it to the Hugging Face Hub
# model.push_to_hub(run_name)

# Evaluate

In [ ]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sparse_encoder.evaluation import (
    SparseInformationRetrievalEvaluator,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 1) Load Italian subsets
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
# params = {"path": "mteb/WikipediaRetrievalMultilingual", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
# params = {"path": "mteb/WebFAQRetrieval", "split": "test"}
# names = ["ita-corpus", "ita-queries", "ita-qrels"]
params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)

# 4) Build evaluator
ir_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    batch_size=4,
    accuracy_at_k=[1],
    write_predictions=True
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)

# 6) Run evaluation
results = ir_evaluator(model, output_path='../results/opensearch-project__opensearch-neural-sparse-encoding-multilingual-v1/predictions')
print(results)

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Corpus Chunks: 100%|██████████| 1/1 [02:56<00:00, 176.66s/it]

{'dot_accuracy@1': 0.425, 'dot_precision@1': 0.425, 'dot_precision@3': 0.18833333333333332, 'dot_precision@5': 0.122, 'dot_precision@10': 0.0665, 'dot_recall@1': 0.425, 'dot_recall@3': 0.565, 'dot_recall@5': 0.61, 'dot_recall@10': 0.665, 'dot_ndcg@10': 0.5425863798128064, 'dot_mrr@10': 0.5035873015873016, 'dot_map@100': 0.5127525880018967, 'query_active_dims': 30.75, 'query_sparsity_ratio': 0.9997095741365143, 'corpus_active_dims': 274.98040771484375, 'corpus_sparsity_ratio': 0.9974028805739114, 'avg_flops': 2.6418862342834473}


## MTEB Benchmark

In [1]:
import yaml
import mteb

with open("../configs/ds/mteg-retrieval-ita.yml", "r") as f:
    config = yaml.safe_load(f)

excluded = set(config["excluded_tasks"])

tasks = mteb.get_tasks(
    languages=config["languages"],
    modalities=config["modalities"],
    task_types=config["task_types"],
    exclusive_language_filter=True,
    eval_splits=["test"],
)

tasks = [t for t in tasks if t.metadata.name not in excluded]

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import mteb

model_names = [
    "mteb/baseline-bm25s",
    "nickprock/splade-bert-base-italian-xxl-uncased-cv",
    "google/embeddinggemma-300m",
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
]
cache = mteb.ResultCache("..")
results = cache.load_results(models=model_names, tasks=tasks, require_model_meta=False)

results.to_dataframe()

model_name,task_name,google/embeddinggemma-300m,mteb/baseline-bm25s,nickprock/splade-bert-base-italian-xxl-uncased-cv,opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
0,MuPLeR-retrieval,0.84454,0.78597,0.510880,0.542586
1,WebFAQRetrieval,0.79523,0.55918,0.517111,0.554244
2,WikipediaRetrievalMultilingual,0.91761,0.84360,0.799550,0.837200


In [4]:
import mteb
from sentence_transformers import SparseEncoder, SentenceTransformer

# model = SparseEncoder(
#     "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# )
cache = mteb.ResultCache("..")
model = SentenceTransformer(
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
)

results = mteb.evaluate(
    model, tasks=tasks, cache=cache, encode_kwargs={"batch_size": 16}
)

Some weights of BertModel were not initialized from the model checkpoint at opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/mteb/models/model_meta.py:681: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/mteb/models/model_meta.py:646: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),
Evaluating task WebFAQRetrieval: 100%|██████████| 2/2 [16:36<00:00, 498.15s/it]


In [3]:
import mteb

# 1. Get the specialized BM25 baseline model
# This uses the 'bm25s' library under the hood
bm25_model = mteb.get_model("mteb/baseline-bm25s")

# 2. Run the evaluation using the MTEB framework
# This will handle the dataset loading for you internally
cache = mteb.ResultCache("..")
results = mteb.evaluate(model=bm25_model, tasks=tasks, cache=cache)

print(results)

Evaluating task WikipediaRetrievalMultilingual: 100%|██████████| 3/3 [02:56<00:00, 58.71s/it]

model_name='mteb/baseline-bm25s' model_revision='0_1_10' task_results=[TaskResult(task_name=MuPLeR-retrieval, main_score=0.79, scores=..., ...), TaskResult(task_name=WebFAQRetrieval, main_score=0.56, scores=..., ...), TaskResult(task_name=WikipediaRetrievalMultilingual, main_score=0.84, scores=..., ...)] default_modalities=['text'] exceptions=[] experiment_name=None


# Post Analysis


When you run the analysis check for these specific markers:

1.  **Vocabulary Mismatch:** Look at the **Query** expansions. If "tutele" didn't expand to "recesso" or "garanzie", the model is purely lexical and failing to bridge the legal gap.
2.  **Language Bleed:** In the **False Positive** expansions, look for tokens like `outside`, `sol`, or non-Latin scripts. If they have high weights, the model is matching the doc based on English "translation" noise rather than Italian legal facts.
3.  **Hallucination:** If the query about "fondi chiusi" (closed-end funds) expands to "porta" or "finestra" because it saw "chiuso" (closed), you have a semantic hallucination where the model took a literal meaning of a financial term.
4.  **Term Saturation:** Check the weights of tokens like `il`, `che`, `ha`. If they are in your Top 15 list with weights similar to `investitore`, the model's **Flop** (the Sparsity/Weighting mechanism) isn't filtering stop-words properly.

In [1]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)


# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
    device='cpu'
)

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [62]:
model._named_members

<bound method Module._named_members of SparseEncoder(
  (0): Router(
    default_route='document'
    (sub_modules): ModuleDict(
      (query): Sequential(
        (0): SparseStaticEmbedding({'frozen': False}, dim=105879, tokenizer=BertTokenizerFast)
      )
      (document): Sequential(
        (0): Transformer({'transformer_task': 'fill-mask', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertForMaskedLM'})
        (1): SpladePooling({'pooling_strategy': 'max', 'activation_function': 'relu', 'embedding_dimension': 768})
      )
    )
  )
)>

In [ ]:
from sentence_transformers import SparseEncoder

model = SparseEncoder("opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1", device='cpu')

sentences = [
    "Il tempo è perfetto oggi.",
    "Che sole c'è fuori!",
    "Ha guidato fino allo stadio.",
    "The patient complained of severe cephalalgia"
]
embeddings = model.encode_document(sentences)
print(embeddings.shape)
# (3, 30522)

# Get the similarity scores for the embeddings
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[   32.4323,     5.8528,     0.0258],
#         [    5.8528,    26.6649,     0.0302],
#         [    0.0258,     0.0302,    24.0839]])

# Let's decode our embeddings to be able to interpret them
decoded = model.decode(embeddings, top_k=10)
for decoded, sentence in zip(decoded, sentences):
    print(f"Sentence: {sentence}")
    print(f"Decoded: {decoded}")
    print()


torch.Size([4, 105879])
tensor([[26.2482,  2.5714,  0.9960,  0.0683],
        [ 2.5714, 25.7051,  0.8058,  0.2033],
        [ 0.9960,  0.8058, 33.9364,  0.2061],
        [ 0.0683,  0.2033,  0.2061, 29.0440]])
Sentence: Il tempo è perfetto oggi.
Decoded: [('tempo', 2.130925178527832), ('time', 1.5420423746109009), ('tiempo', 1.4455000162124634), ('oggi', 1.2541877031326294), ('##fetto', 1.2526025772094727), ('per', 1.245772361755371), ('temps', 1.0007442235946655), ('tuttora', 0.8932561874389648), ('ormai', 0.7772701382637024), ('##fet', 0.7290021181106567)]

Sentence: Che sole c'è fuori!
Decoded: [('fuori', 1.8266558647155762), ('sole', 1.7858755588531494), ('خارج', 1.4555096626281738), ('sol', 1.2063599824905396), ('बाहर', 1.0855473279953003), ('sola', 1.045993685722351), ('che', 0.9845602512359619), ('outside', 0.9647106528282166), ('dentro', 0.8923901319503784), ('بیرون', 0.8822744488716125)]

Sentence: Ha guidato fino allo stadio.
Decoded: [('guida', 1.331709623336792), ('stadio', 

### nickprock/splade-bert-base-italian-xxl-uncased-cv
*The Monolingual Specialist*

```
Sentence: Il tempo è perfetto oggi.
Decoded: [('oggi', 1.0779329538345337), ('tempo', 1.0449038743972778), ('meteo', 0.8066657781600952), ('ieri', 0.6896640062332153), ('perfetto', 0.5086107850074768), ('martedi', 0.4426819682121277), ('stasera', 0.3519502282142639), ('domani', 0.19588139653205872), ('il', 0.15203651785850525), ('venerdi', 0.10040995478630066)]

Sentence: Che sole c'è fuori!
Decoded: [('sole', 1.4501569271087646), ('fuori', 0.22559906542301178), ('comunque', 0.05879998207092285)]

Sentence: Ha guidato fino allo stadio.
Decoded: [('fino', 0.6662643551826477), ('stadio', 0.21094673871994019)]
```


* **The Good:**
    * **Deep Semantic Expansion:** In the first sentence, it correctly identifies that "tempo" (time/weather) refers to the weather by expanding to **"meteo"**. 
    * **Temporal Logic:** It expands "oggi" (today) to other temporal markers like "ieri" (yesterday) and "martedi" (Tuesday). While technically incorrect as a literal translation, this is great for retrieval because it clusters "weather-talk" together.
    * **High Precision:** It stays strictly within the Italian language, preventing "cross-lingual noise."
* **The Bad:**
    * **Extreme Sparsity:** In the second and third sentences, it is almost *too* quiet. 
    * **Failure on Verbs:** In "Ha guidato fino allo stadio," it completely ignored the verb **"guidato"** (driven). If a user searches for "guida" (driving), this model would likely fail to retrieve this sentence.
* **Peculiarities:** * It behaves more like a keyword extractor with a few "thesaurus" hits than a full neural expander.

This model is "safe" but lacks "imagination." It is excellent for tasks where precision is king and you don't want weird cross-language interference. However, its failure to capture the verb "guidato" suggests it might suffer from significant **"False Negatives"** (missing relevant documents because the expansion was too conservative).

### opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
*The Multilingual Powerhouse*

```
Sentence: Il tempo è perfetto oggi.
Decoded: [('tempo', 2.130925178527832), ('time', 1.5420420169830322), ('tiempo', 1.4455000162124634), ('oggi', 1.254187822341919), ('##fetto', 1.2526030540466309), ('per', 1.2457724809646606), ('temps', 1.0007436275482178), ('tuttora', 0.893256425857544), ('ormai', 0.7772701382637024), ('##fet', 0.7290024757385254)]

Sentence: Che sole c'è fuori!
Decoded: [('fuori', 1.8266568183898926), ('sole', 1.7858753204345703), ('خارج', 1.4555113315582275), ('sol', 1.2063595056533813), ('बाहर', 1.0855478048324585), ('sola', 1.0459933280944824), ('che', 0.9845602512359619), ('outside', 0.964711606502533), ('dentro', 0.8923905491828918), ('بیرون', 0.8822756409645081)]

Sentence: Ha guidato fino allo stadio.
Decoded: [('guida', 1.331709384918213), ('stadio', 1.169238805770874), ('감독', 1.1202867031097412), ('stadium', 1.1130553483963013), ('manager', 1.111269235610962), ('dirige', 1.0480386018753052), ('estadio', 1.0043498277664185), ('ha', 0.9999247789382935), ('fino', 0.9253085851669312), ('coach', 0.8294572830200195)]
```

* **The Good:**
    * **Robust Recall:** It captures the root of verbs (e.g., **"guida"** for "guidato"), which is vital for effective retrieval.
    * **Conceptual Depth:** In the third sentence, it associates "stadio" and "guida" with **"manager"** and **"coach"**. It understands the *domain* of the sentence (sports/professional) rather than just the words.
* **The Bad:**
    * **Subword Sensitivity:** It uses subwords (e.g., **"##fetto"**) to maintain a high weight on the original terms.
    * **Language Bleed:** It expands "sole" into Arabic, Hindi, and English. If your corpus is purely Italian, these tokens are "dead weight" that take up index space without adding value.
    * **Antonym Drift:** It expands "fuori" (outside) to **"dentro"** (inside). In retrieval, this can lead to "false positives" where you search for "outdoor activities" and get "indoor" results.
* **Peculiarities:** * It effectively translates the query into a "universal concept" space. It doesn't care that you spoke Italian; it wants to find the *idea* of a sun/sole/sunlight.

Despite the "messiness" of the multilingual expansions, this is the superior model for general Information Retrieval. Its ability to stem "guidato" into "guida" and its aggressive conceptual expansion ("stadium" $\rightarrow$ "coach") ensures a much higher **Recall**. In a vector database (like OpenSearch or Pinecone), those extra multilingual tokens usually don't hurt as much as a missing primary keyword.


### Identify "Low Performers"
This function reads your predictions file and calculates Recall@k for every query, returning the ones where the model failed completely ($Recall = 0$).

In [41]:
import json
import math
import pandas as pd


def dcg_at_k(relevances, k):
    """
    DCG@k using the standard formulation:
    sum_i rel_i / log2(i + 2)

    i is zero-based, so rank 1 has denominator log2(2).
    """
    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevances[:k])
    )


def ndcg_at_k(retrieved_ids, gold_ids, k):
    """
    Binary nDCG@k:
    - relevance = 1 if retrieved doc is in gold_ids
    - relevance = 0 otherwise
    """
    gold_ids = set(gold_ids)

    if not gold_ids:
        return 0.0

    retrieved_relevances = [
        1 if cid in gold_ids else 0
        for cid in retrieved_ids[:k]
    ]

    dcg = dcg_at_k(retrieved_relevances, k)

    ideal_relevances = [1] * min(len(gold_ids), k)
    idcg = dcg_at_k(ideal_relevances, k)

    return dcg / idcg if idcg > 0 else 0.0


def get_performers(predictions_path, relevant_docs, k=10, target_ndcg=0):
    performers = []

    with open(predictions_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)

            qid = str(data["query_id"])

            # Normalize prediction IDs to strings
            results = data["results"]
            top_results = results[:k]

            retrieved_ids = [str(res["corpus_id"]) for res in top_results]
            retrieved_scores = [res["score"] for res in top_results]

            # Normalize gold IDs to strings
            gold_ids = sorted(str(cid) for cid in relevant_docs.get(qid, set()))

            ndcg = ndcg_at_k(
                retrieved_ids=retrieved_ids,
                gold_ids=gold_ids,
                k=k,
            )

            # Optional: model scores for gold docs, if they appear anywhere in the prediction file
            all_scores_lookup = {
                str(res["corpus_id"]): res["score"]
                for res in results
            }

            gold_scores = [
                all_scores_lookup.get(cid, None)
                for cid in gold_ids
            ]

            performers.append({
                "qid": qid,
                "query": data["query"],
                f"ndcg_at_{k}": ndcg,
                "retrieved_ids": retrieved_ids,
                "retrieved_scores": retrieved_scores,
                "gold_ids": gold_ids,
                "gold_scores": gold_scores,
            })

    df = pd.DataFrame(performers)

    return df[df[f"ndcg_at_{k}"] == target_ndcg].sort_values(by="qid")

# Usage:
df_performers = get_performers("/home/lpozzi/Git/lm-toolkit/results/opensearch-project__opensearch-neural-sparse-encoding-multilingual-v1/predictions/MuPLeR-retrieval/Information-Retrieval_evaluation_predictions_dot.jsonl", relevant_docs, target_ndcg=0)
df_performers

,qid,query,ndcg_at_10,retrieved_ids,retrieved_scores,gold_ids,gold_scores
67,05e24457-4e76-4ce9-8baf-53f988f5cc4c,Quale bulbo di regione delimitata è classifica...,0.0,"[7362, 4494, 9014, 7150, 7852, 5667, 5775, 800...","[4.72084903717041, 4.481014728546143, 4.332448...",[7074],[3.5849926471710205]
158,07d4fd87-bfc7-4ae6-8bd3-ae5b1fd05607,Perché esortare gli Stati membri a stanziare i...,0.0,"[6257, 5956, 4133, 740, 1422, 5960, 2805, 9845...","[4.6294708251953125, 4.625259876251221, 4.2867...",[4387],[3.567355155944824]
160,0aa5e8f3-4f5d-43d6-bf31-320d9396c80b,Quale schema d'intervento a tre livelli è prop...,0.0,"[3992, 4855, 2043, 6141, 7996, 2042, 7415, 215...","[5.8867387771606445, 5.773351669311523, 5.6208...",[3283],[None]
125,10feeabc-7fb6-49a6-a749-69d31964ba79,Quale organismo consultivo dell'UE ha accolto ...,0.0,"[6255, 709, 710, 878, 6768, 1702, 4861, 8682, ...","[7.848711013793945, 6.677041530609131, 5.66589...",[6770],[4.616129398345947]
150,11430392-723c-4e0f-a43c-a9f0d3845cbb,Quale autorizzazione istituzionale devono aver...,0.0,"[5956, 6169, 6521, 2253, 1750, 1745, 2908, 626...","[4.502736568450928, 4.044743537902832, 3.97131...",[7534],[None]
...,...,...,...,...,...,...,...
3,eaf58e78-e2b7-43ec-b417-4106d5a6eea3,Quale organismo dovrebbe essere istituzionalme...,0.0,"[8585, 6406, 3295, 14, 9995, 8894, 3796, 8710,...","[6.3113298416137695, 6.193119525909424, 5.9672...",[8583],[5.593110084533691]
147,ec920fca-8c31-4c85-8fb2-423248bf7e71,Quale Paese non ha adottato misure nazionali p...,0.0,"[9171, 9169, 9014, 6406, 9168, 885, 6242, 5130...","[8.384749412536621, 7.006247520446777, 6.94484...",[1317],[None]
111,eeb8ccc3-a82e-4abd-9e5d-6688f8646e54,Quale organo esecutivo UE svilupperà specifich...,0.0,"[9383, 6352, 7891, 4407, 3948, 8831, 4076, 877...","[7.768490314483643, 5.84207820892334, 5.164144...",[8836],[3.726895332336426]
15,f82a0c60-b4a8-4261-a07b-300be0df8c2d,Quale modello analitico del 1983 è criticato p...,0.0,"[5960, 7996, 9929, 2122, 6257, 5682, 9993, 595...","[6.664707660675049, 6.08540678024292, 5.891228...",[8609],[None]


In [45]:
df_performers['retrieved_scores'].iloc[0]

[4.72084903717041,
 4.481014728546143,
 4.332448482513428,
 4.2951884269714355,
 4.17866849899292,
 4.146534442901611,
 4.1106791496276855,
 4.065140724182129,
 4.049183368682861,
 4.036776542663574]

### Qualitative Inspection (FP vs FN)
Once you have a failure ID, this function prints a "Trial" view: what the model liked vs. what it should have liked.

In [ ]:
def qualitative_inspection(qid, df_fail, corpus):
    row = df_fail[df_fail['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}\n")
    print(f"❌ FALSE POSITIVES (Top 3 Model Choices):")
    for cid in row['retrieved_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")
        
    print(f"\n✅ FALSE NEGATIVES (What the model missed):")
    if not row['gold_ids']:
        print("  - No ground truth defined for this query.")
    for cid in row['gold_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")

# Usage:
qualitative_inspection("05e24457-4e76-4ce9-8baf-53f988f5cc4c", df_performers, corpus)

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362]: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Dirett...
  - [ID 4494]: Per evitare ulteriori danneggiamenti che comporterebbero un calo di quantità di prodotto vendibile e soprattutto un calo di qualità dell'intera partita è determinante eseguire tale lavorazione in temp...
  - [ID 9014]: Nella pianificazione delle TEN-E si dovrebbe assegnare un mandato chiaro all'ENTSO-E e all'ACER nonché definire il ruolo di mediazione dell'UE. Il Libro verde non è sufficientemente esplicito su quest...

✅ FALSE NEGATIVES (What the model missed):
  - [ID 7074]: Le varietà Makói vöröshagyma o Makói hagyma sono classificate, imballate ed etichettat

### SPLADE Expansion Diagnosis
To see why a False Positive was ranked so high, we need to see the activated tokens. This function maps the sparse vector back to the vocabulary.

In [2]:
import torch

# 4. Map indices to words using the mlm_transformer tokenizer
VOCAB = {v: k for k, v in mlm_transformer.tokenizer.get_vocab().items()}

def diagnose_expansion(text, model, mode="query", top_n=-1):
    """
    Updated to use SparseEncoder's query and document specific encoding logic.
    """
    # 1. Run inference based on mode
    with torch.no_grad():
        if mode == "query":
            # SparseEncoder.encode_query returns a tensor (batch_size, vocab_size)
            vector = model.encode_query([text], convert_to_sparse_tensor=False)[0]
        else:
            # SparseEncoder.encode_document returns a tensor (batch_size, vocab_size)
            vector = model.encode_document([text], convert_to_sparse_tensor=False)[0]
    
    # 3. Move to CPU and convert to list
    weights = vector.cpu().tolist()
    
    activated = []
    for i, w in enumerate(weights):
        if w > 0: # Small epsilon to avoid float noise
            token = VOCAB.get(i, f"[UNK_{i}]")
            activated.append((token, w))
            
    # 5. Sort by weight descending
    sorted_activated = sorted(activated, key=lambda x: x[1], reverse=True)
    
    if top_n == -1:
        return sorted_activated
    return sorted_activated[:top_n]

def format_expansion(activations):
    """Formats the output of diagnose_expansion into a clean list."""
    for word, weight in activations:
        # Aligns word to the left (15 chars) and weight to 4 decimal places
        print(f"         {word:<15} | {weight:.4f}")

def show_intersection(query_weights, doc_weights, top_k=-1):
    """
    Fixed version: Converts list of tuples to dicts internally to handle 
    the intersection and weight lookups properly.
    """
    # Convert lists of (token, weight) tuples into dictionaries for lookup
    q_dict = dict(query_weights)
    d_dict = dict(doc_weights)
    
    # Find common tokens using set intersection on the dictionary keys
    common_tokens = set(q_dict.keys()) & set(d_dict.keys())
    
    # Build the intersection data
    intersection = {k: (q_dict[k], d_dict[k]) for k in common_tokens}
    
    # Sort by the product (contribution to total score)
    sorted_inter = sorted(intersection.items(), key=lambda x: x[1][0] * x[1][1], reverse=True)
    
    print(f"    Intersection ({len(sorted_inter)} tokens):")
    if not sorted_inter:
        print("      [No Overlap]")
    else:
        # Handle the top_k slicing (if -1, show everything)
        display_list = sorted_inter if top_k == -1 else sorted_inter[:top_k]
        
        for token, (qw, dw) in display_list:
            # We also calculate the product to show the actual impact on retrieval
            product = qw * dw
            print(f"      - {token:<15} | Q: {qw:.4f} * D: {dw:.4f} = Score: {product:.4f}")


def qualitative_expansion_inspection(qid, df_performers, corpus, queries, model):
    row = df_performers[df_performers['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}")
    # query branch is not using SPLADE pooling
    #     token appears in query -> 1.0
    #     token not in query     -> 0.0
    query_expansion = diagnose_expansion(queries[qid], model, mode="query", top_n=-1)
    # format_expansion(query_expansion[:15]) # Show top 15 for query clarity

    # 1. Identify True Positives (Gold IDs that actually appear in Retrieved IDs)
    tps = [cid for cid in row['gold_ids'] if cid in row['retrieved_ids']]
    if tps:
        print(f"\n🚀 TRUE POSITIVES:")
        for cid in tps:
            rank = row['retrieved_ids'].index(cid) + 1
            score = row['retrieved_scores'][rank-1]
            print(f"  - [ID {cid}] | Rank: {rank} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)

    # 2. False Positives (Top of the list, but not in gold)
    print(f"\n❌ FALSE POSITIVES (Top 3 Model Choices):")
    for i, cid in enumerate(row['retrieved_ids'][:3]):
        if cid not in row['gold_ids']:
            score = row['retrieved_scores'][i]
            print(f"  - [ID {cid}] | Rank: {i+1} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)
        
    # 3. False Negatives (Gold IDs that did NOT make it into the top k)
    cid  = row['gold_ids'][0]
    if cid not in row['retrieved_ids']:
        print(f"\n✅ FALSE NEGATIVES (What the model missed):")
        score = row['gold_scores'][0]
        print(f"  - [ID {cid}] | Score: {score:.4f} (Below Retrieval Threshold)")
        print(f"    Text: {corpus[cid][:500]}...")
        # Fixed: Define doc_expansion BEFORE showing intersection
        doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
        show_intersection(query_expansion, doc_expansion, top_k=10)
    else:
        print(f"\n✅ Zero FALSE NEGATIVES.")


import json
from pathlib import Path


def write_expansion_json(
    texts,
    model,
    model_name,
    output_path,
    mode="query",
    top_n=-1,
    ensure_ascii=False,
):
    output_path = Path(output_path)

    rows = []
    for idx, text in enumerate(texts):
        expansion = diagnose_expansion(
            text=text,
            model=model,
            mode=mode,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": float(weight),
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {
        model_name: rows
    }

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

In [43]:
for target_qid in df_performers['qid'].iloc[:1].tolist():
    qualitative_expansion_inspection(
        qid=target_qid,
        df_performers=df_performers,
        corpus=corpus,
        queries=queries,
        model=model,
    )

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362] | Rank: 1 | Score: 4.7208
    Text: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Direttive, sono però conformi al quadro giuridico vigente, esiste pur sempre la possibilità di una violazione, magari inconsapevole, di tale normativa. Il CESE raccomanda dunque ai committenti di esaminare attentamente l'allegato e di seguirne scrupolosamente le raccomandazioni. Nel caso in cui l'amminist...
    Intersection (5 tokens):
      - ##e             | Q: 1.0000 * D: 2.0762 = Score: 2.0762
      - deli            | Q: 1.0000 * D: 1.2698 = Score: 1.2698
      - ##o             | Q: 1.0000 * D: 0.6573 = Score: 0.6573
      - ##to            | Q

In [47]:
queries

{'ba404e28-77f6-4dc0-8c45-b54c22053c83': 'Come si estendono le tutele fuori locali commerciali quando un investitore al dettaglio aderisce a fondo immobiliare chiuso per investire?',
 '274beb3a-3ded-48f5-893c-b77181f358c9': 'Perché le imprese sottofinanziano la formazione se i benefici sociali superano i ritorni privati dalle competenze trasferibili?',
 '776d33f1-4283-40cb-9463-d4758a5a1ee7': 'Chi mantiene il potere di firma congiunta sul conto fruttifero a doppia firma per assicurare la corretta erogazione?',
 'eaf58e78-e2b7-43ec-b417-4106d5a6eea3': "Quale organismo dovrebbe essere istituzionalmente separato e dotato dei poteri per imporre regolarmente misure affinché un'emittente adempia al servizio pubblico?",
 '5c849642-6405-4e6a-86ad-8a103649c485': 'Chi ha concluso che investitori buyout rafforzano il controllo e tagliano personale e stabilimenti sottoperformanti trattando pragmaticamente le trattative sindacali?',
 'cbd0e5c6-70f8-4f6d-b1cf-9b1dd5b7deeb': 'Quale organismo fornì un

In [51]:
print(queries['ba404e28-77f6-4dc0-8c45-b54c22053c83'])
diagnose_expansion('Il medico ha prescritto un farmaco per ridurre la pressione alta.', model, mode="doc", top_n=-1)

Come si estendono le tutele fuori locali commerciali quando un investitore al dettaglio aderisce a fondo immobiliare chiuso per investire?


[('alta', 1.5419769287109375),
 ('farm', 1.3651257753372192),
 ('pressione', 1.3356409072875977),
 ('pres', 1.1909023523330688),
 ('rid', 1.1308432817459106),
 ('presion', 1.1208887100219727),
 ('##aco', 1.0940639972686768),
 ('alto', 1.0501487255096436),
 ('medico', 1.0243804454803467),
 ('haute', 0.9650628566741943),
 ('##ضغط', 0.9495495557785034),
 ('pression', 0.9311445951461792),
 ('cure', 0.9037303924560547),
 ('medicina', 0.9012002944946289),
 ('druck', 0.881564199924469),
 ('pressure', 0.8752270340919495),
 ('medica', 0.8582887649536133),
 ('فشار', 0.8335608243942261),
 ('tekanan', 0.815278172492981),
 ('cura', 0.7890410423278809),
 ('tratamiento', 0.763865053653717),
 ('tiba', 0.7505817413330078),
 ('درمان', 0.7021135091781616),
 ('treatment', 0.6995443105697632),
 ('pressao', 0.6894218325614929),
 ('tratar', 0.6880136132240295),
 ('for', 0.6726405024528503),
 ('reduce', 0.6724516153335571),
 ('hyper', 0.6702060699462891),
 ('##irat', 0.6388790011405945),
 ('##ac', 0.628410875

In [3]:
sentences = [
    "Il medico ha prescritto un farmaco per ridurre la pressione alta.",
    "Dopo l’incidente, l’auto è stata portata dal meccanico.",
    "Il ragazzo ha comprato un nuovo telefono perché il vecchio si era rotto.",
    "La banca ha approvato il prestito per l’acquisto della casa.",
    "Il cane correva felice nel parco durante una giornata di sole.",
    "La squadra ha perso la partita nonostante una prestazione eccellente.",
    "Non ho detto che Marco abbia rubato il portafoglio.",
    "Questo ristorante è una perla nascosta nel centro storico.",
    "Il professore ha spiegato la teoria della relatività con esempi semplici.",
    "La consegna del pacco è stata rimandata a causa dello sciopero dei trasporti.",
]

write_expansion_json(
    texts=sentences,
    model=model,
    model_name='splade-bert-base-italian-xxl-uncased-cv',
    output_path="splade_expansions.json",
    mode="doc",
    top_n=50,
)